# Tutorial 05 -- Op³ → OpenFAST SoilDyn export

Plug any Op³ foundation into a production OpenFAST v5 coupled simulation.

## 1. Build a PISA foundation for the Gunsan tripod

In [ ]:
from op3.standards.pisa import SoilState, pisa_pile_stiffness_6x6

profile = [
    SoilState(0.0,  5.0e7, 35, 'sand'),
    SoilState(15.0, 1.0e8, 35, 'sand'),
    SoilState(36.0, 1.5e8, 36, 'sand'),
]
K = pisa_pile_stiffness_6x6(diameter_m=6.0, embed_length_m=36.0, soil_profile=profile)
print('PISA-derived 6x6 stiffness:')
print(f'  Kxx    = {K[0,0]:.3e} N/m')
print(f'  Krxrx  = {K[3,3]:.3e} Nm/rad')
print(f'  Kxrx   = {K[0,4]:.3e} N')

## 2. Export to SoilDyn .dat format

In [ ]:
from op3.openfast_coupling.soildyn_export import write_soildyn_from_pisa

out = write_soildyn_from_pisa(
    'demo_SoilDyn.dat',
    diameter_m=6.0, embed_length_m=36.0,
    soil_profile=profile,
    location_xyz=(-24.80, 0.0, -45.0),  # SubDyn joint 1 of OC3 Tripod
)
print(f'Wrote: {out}')
print('\nFirst 20 lines:')
print('\n'.join(out.read_text().splitlines()[:20]))

## 3. Patch your .fst to use SoilDyn

```text
1   CompSoil   - Compute soil-structural dynamics {1=SoilDyn}
"demo_SoilDyn.dat"   SoilFile
```

Then run OpenFAST:
```bash
tools/openfast/OpenFAST.exe your_deck.fst
```

On the Gunsan v5 deck this produces a normal-terminating 5-second simulation with all 8 modules (ElastoDyn + InflowWind + AeroDyn + ServoDyn + SeaState + HydroDyn + SubDyn + SoilDyn with Op³ PISA K). Wall time ~1.9 minutes.

## 4. Clean up

In [ ]:
import os
os.unlink('demo_SoilDyn.dat')